<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:

import pennylane as qml
import pennylane.numpy as np
from scipy.optimize import minimize

symbols = ["H", "H", "H"]

def h3_ground_energy(bond_length):
    """
    Uses VQE to calculate the ground energy of the H3+ molecule with the given bond length.

    Args:
        - bond_length(float): The bond length of the H3+ molecule modelled as an
        equilateral triangle.

    Returns:
        - float: The ground state energy of the H3+ molecule at the given bond length.
    """
    # Coordinates for an equilateral triangle of side bond_length (units: Angstrom)
    a = bond_length
    geometry = np.array([
        [0.0, 0.0, 0.0],
        [a, 0.0, 0.0],
        [a/2, a * np.sqrt(3)/2, 0.0]
    ])

    charge = 1   # H3+ ion, so charge is +1
    mult = 1     # Singlet

    # Build molecular Hamiltonian in the STO-3G minimal basis
    H, n_qubits = qml.qchem.molecular_hamiltonian(
        symbols,
        geometry,
        charge=charge,
        mult=mult,
        basis="sto-3g",
    )

    n_electrons = 2  # 3 hydrogens = 3 electrons, charge +1 => 2 electrons

    hf_state = qml.qchem.hf_state(n_electrons, n_qubits)
    singles, doubles = qml.qchem.excitations(n_electrons, n_qubits)
    excitations = singles + doubles

    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def circuit(params):
        qml.BasisState(hf_state, wires=range(n_qubits))
        for i, ex in enumerate(excitations):
            if len(ex) == 2:
                qml.SingleExcitation(params[i], wires=ex)
            elif len(ex) == 4:
                qml.DoubleExcitation(params[i], wires=ex)
        return qml.expval(H)

    # Initial guess: all zero
    params0 = np.zeros(len(excitations))

    result = minimize(lambda x: circuit(x), params0, method="BFGS", tol=1e-6, options={'maxiter': 300})
    energy = result.fun
    return float(np.real(energy))


# ========= TESTS =========

def test_h3_15():
    out = h3_ground_energy(1.5)
    assert abs(out - (-1.232574)) < 1e-4

def test_h3_08():
    out = h3_ground_energy(0.8)
    assert abs(out - (-0.3770325)) < 1e-4

if __name__ == "__main__":
    test_h3_15()
    test_h3_08()
    print("Success!")


AttributeError: module 'pennylane.optimize' has no attribute 'minimize'